# DX 704 Week 1 Project

This week's project will build a portfolio risk and return model, and make investing recommendations for hypothetical clients.
You will collect historical data, estimate returns and risks, construct efficient frontier portfolios, and sanity check the certainty of the maximum return portfolio.

The full project description and a template notebook are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-01


Feel free to use optimization tools or libraries (such as CVXOPT or scipy.optimize) to perform any calculations required for this mini project.

### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [58]:
pip install numpy pandas matplotlib seaborn scikit-learn cvxpy yfinance


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Part 1: Collect Data

Collect historical monthly price data for the last 24 months covering 6 different stocks.
The data should cover 24 consecutive months including the last month that ended before this week's material was released on Blackboard.
To be clear, if a month ends between the Blackboard release and submitting your project, you do not need to add that month.

The six different stocks must include AAPL, SPY and TSLA.
At least one of the remaining 3 tickers must start with the same letter as your last name (e.g. professor Considine could use COIN).
This is to encourage diversity in what stocks you analyze; if you discuss this project with classmates, please make sure that you pick different tickers to differentiate your work.
Do not pick stocks with fewer than 24 consecutive months of price data.

In [59]:
import numpy as np
import pandas as pd
import cvxpy as cp
import yfinance as yf

TICKERS = ["AAPL", "SPY", "TSLA", "MDA.TO", "BA", "LMT"]

In [60]:
LAST_MONTH = "2026-07"

period = pd.Period(LAST_MONTH, freq="M")
raw = yf.download(TICKERS,
                  start=(period - 25).start_time,
                  end=period.end_time + pd.Timedelta(days=1),
                  auto_adjust=True, progress=False)

close = raw["Close"][TICKERS]

monthly = close.groupby([close.index.year, close.index.month]).tail(1)
monthly = monthly[monthly.index <= period.end_time]

historical_prices = monthly.tail(24)
historical_prices.index.name = "date"

print(historical_prices.shape)
print(historical_prices.isna().sum())
print(historical_prices.index[0], "->", historical_prices.index[-1])
historical_prices

(24, 6)
Ticker
AAPL      0
SPY       0
TSLA      0
MDA.TO    0
BA        0
LMT       0
dtype: int64
2024-08-30 00:00:00 -> 2026-07-31 00:00:00


Ticker,AAPL,SPY,TSLA,MDA.TO,BA,LMT
date,,,,,,
2024-08-30,227.100266,550.644348,214.110001,15.850000,173.740005,535.652222
2024-09-30,231.067093,562.210449,261.630005,17.379999,152.039993,554.245422
2024-10-31,224.035889,557.193542,249.850006,21.299999,149.309998,517.732422
2024-11-29,235.620132,590.420898,345.160004,26.910000,155.440002,501.955322
2024-12-31,248.615799,576.215393,403.839996,29.530001,177.000000,463.629578
2025-01-31,234.299683,591.690369,404.600006,23.059999,176.520004,441.695129
2025-02-28,240.361588,584.178955,292.980011,23.209999,174.630005,429.692688
2025-03-31,220.772079,551.628906,259.160004,27.490000,170.550003,429.346680
2025-04-30,211.200943,546.846191,282.160004,26.910000,183.240005,459.180176


Save the data as a TSV file named "historical_prices.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
The date should be the last trading day of the month, so it may not be the last day of the month.
For example, the last trading day of November 2024 was 2024-11-29.
The remaining columns should contain the adjusted closing prices of the corresponding stock tickers on that day.


In [61]:
# YOUR CHANGES HERE

out = historical_prices.copy()
out.index = out.index.strftime("%Y-%m-%d")
out.to_csv("historical_prices.tsv", sep="\t")

Submit "historical_prices.tsv" in Gradescope.

## Part 2: Calculate Historical Asset Returns

Calculate the historical asset returns based on the price data that you previously collected.

In [62]:
historical_returns = historical_prices.pct_change().dropna()
print(historical_returns.shape)
historical_returns

(23, 6)


Ticker,AAPL,SPY,TSLA,MDA.TO,BA,LMT
date,,,,,,
2024-09-30,0.017467,0.021005,0.221942,0.096530,-0.124899,0.034711
2024-10-31,-0.030429,-0.008924,-0.045025,0.225547,-0.017956,-0.065879
2024-11-29,0.051707,0.059633,0.381469,0.263380,0.041056,-0.030473
2024-12-31,0.055155,-0.024060,0.170008,0.097362,0.138703,-0.076353
2025-01-31,-0.057583,0.026856,0.001882,-0.219099,-0.002712,-0.047310
2025-02-28,0.025872,-0.012695,-0.275877,0.006505,-0.010707,-0.027174
2025-03-31,-0.081500,-0.055719,-0.115435,0.184403,-0.023364,-0.000805
2025-04-30,-0.043353,-0.008670,0.088748,-0.021099,0.074406,0.069486
2025-05-30,-0.053584,0.062845,0.227885,0.054998,0.131412,0.009691


Save the data as a TSV file named "historical_returns.tsv" and include a header row with the column names "date" and the 6 stock ticker symbols.
Each row should have the date at the end of the month and the corresponding *relative* price changes.
For example, if the previous price was \$100 and the new price is \$110, the return value should be 0.10.
There should only be 23 rows of data in this file, since they are computed as the differences of 24 prices.

In [63]:
# YOUR CHANGES HERE

out = historical_returns.copy()
out.index = out.index.strftime("%Y-%m-%d")
out.to_csv("historical_returns.tsv", sep="\t")

Submit "historical_returns.tsv" in Gradescope.

## Part 3: Estimate Returns

Estimate the expected returns for each asset using the previously calculated return data.
Just compute the average (mean) return for each asset over your data set; do not use other estimators that have been mentioned.
This will serve as your estimate of expected return for each asset.

In [64]:
# YOUR CHANGES HERE

estimated_returns = historical_returns.mean().rename("estimated_return")
estimated_returns.index.name = "asset"
estimated_returns

asset
AAPL      0.015299
SPY       0.013994
TSLA      0.028766
MDA.TO    0.062127
BA        0.012737
LMT       0.007473
Name: estimated_return, dtype: float64

Save the estimated returns in a TSV file named "estimated_returns.tsv" and include a header row with the column names "asset" and "estimated_return".

In [65]:
# YOUR CHANGES HERE

estimated_returns.to_csv("estimated_returns.tsv", sep="\t")

Submit "estimated_returns.tsv" in Gradescope.

## Part 4: Estimate Risk

Estimate the covariance matrix for the asset returns to understand how the assets move together.

In [66]:
estimated_covariance = historical_returns.cov()
estimated_covariance.index.name = None
estimated_covariance.columns.name = None
estimated_covariance

,AAPL,SPY,TSLA,MDA.TO,BA,LMT
AAPL,0.004009,0.001047,0.002768,0.001141,0.000119,0.000070
SPY,0.001047,0.001374,0.002789,0.001884,0.001120,-0.000278
TSLA,0.002768,0.002789,0.026752,0.005912,0.002088,0.000166
MDA.TO,0.001141,0.001884,0.005912,0.040407,0.006566,0.002540
BA,0.000119,0.001120,0.002088,0.006566,0.006793,0.000370
LMT,0.000070,-0.000278,0.000166,0.002540,0.000370,0.009181


Save the estimated covariances to a TSV file named "estimated_covariance.tsv".
The header row should have a blank column name followed by the names of the assets.
Each data row should start with the name of an asset for that row, and be followed by the individual covariances corresponding to that row and column's assets.
(This is the format of pandas's `to_csv` method with `sep="\t"` when used on a covariance matrix as computed in the examples.)

In [67]:
# YOUR CHANGES HERE

estimated_covariance.to_csv("estimated_covariance.tsv", sep="\t")

Submit "estimated_covariance.tsv" in Gradescope.

## Part 5: Construct the Maximum Return Portfolio

Compute the maximum return portfolio based on your previously estimated risks and returns.

In [68]:
# YOUR CHANGES HERE

n = len(estimated_returns)
mu = estimated_returns.to_numpy()
Sigma = estimated_covariance.to_numpy()

x_max = cp.Variable(n)
prob_max = cp.Problem(cp.Maximize(mu.reshape(1, -1) @ x_max),
                      [0 <= x_max, cp.sum(x_max) == 1])
estimated_return_maximum = prob_max.solve()

w = np.clip(x_max.value, 0, None)
w = w / w.sum()
maximum_return = pd.Series(w, index=estimated_returns.index, name="allocation")

print(estimated_return_maximum)
maximum_return.round(4)

0.06212738707790342


asset
AAPL      0.0
SPY       0.0
TSLA      0.0
MDA.TO    1.0
BA        0.0
LMT       0.0
Name: allocation, dtype: float64

Save the maximum return portfolio in a TSV file named "maximum_return.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [69]:
# YOUR CHANGES HERE

maximum_return.to_csv("maximum_return.tsv", sep="\t")

Submit "maximum_return.tsv" in Gradescope.

## Part 6: Construct the Minimum Risk Portfolio

Compute the minimum risk portfolio based on your previously estimated risks.

In [70]:
# YOUR CHANGES HERE
x_min = cp.Variable(n)
prob_min = cp.Problem(cp.Minimize(cp.quad_form(x_min, cp.psd_wrap(Sigma))),
                      [0 <= x_min, cp.sum(x_min) == 1])
variance_minimum_risk = prob_min.solve()
estimated_return_minimum_risk = float(x_min.value @ mu)

w = np.clip(x_min.value, 0, None)
w = w / w.sum()
minimum_risk = pd.Series(w, index=estimated_returns.index, name="allocation")

print("variance:", variance_minimum_risk)
print("return:", estimated_return_minimum_risk)
minimum_risk.round(4)

variance: 0.0011077998065453577
return: 0.013132893555688101


asset
AAPL      0.0766
SPY       0.7517
TSLA      0.0000
MDA.TO    0.0000
BA        0.0301
LMT       0.1416
Name: allocation, dtype: float64

Save the minimum risk portfolio in a TSV file named "minimum_risk.tsv".
The header row should have two columns, "asset" and "allocation".
The allocation values should sum up to one.


In [71]:
# YOUR CHANGES HERE

minimum_risk.to_csv("minimum_risk.tsv", sep="\t")

Submit "minimum_risk.tsv" in Gradescope.

## Part 7: Build Efficient Frontier Portfolios

Compute 101 portfolios along the mean-variance efficient frontier with evenly spaced estimated returns.
The first portfolio should be the minimum risk portfolio from part 4, and the last portfolio should be the maximum return portfolio from part 3.
The estimated return of each portfolio should be higher than the previous by one percent of the difference between the first and last portfolios.
That is, the estimated return of the portfolios should be similar to `np.linspace(min_risk_return, max_return, 101)`.


In [72]:
# YOUR CHANGES HERE

targets = np.linspace(estimated_return_minimum_risk, estimated_return_maximum, 101)

rows = []
for i, r in enumerate(targets):
    x_r = cp.Variable(n)
    prob_r = cp.Problem(cp.Minimize(cp.quad_form(x_r, cp.psd_wrap(Sigma))),
                        [0 <= x_r, cp.sum(x_r) == 1, mu @ x_r == r])
    var_r = prob_r.solve()
    w = np.clip(x_r.value, 0, None)
    w = w / w.sum()
    rows.append([i, r, np.sqrt(max(var_r, 0.0))] + list(w))

efficient_frontier = pd.DataFrame(rows, columns=["index", "return", "risk"] + TICKERS)

print(efficient_frontier.shape)  
print(efficient_frontier["risk"].is_monotonic_increasing) 
efficient_frontier.head()

(101, 9)
True


,index,return,risk,AAPL,SPY,TSLA,MDA.TO,BA,LMT
0,0,0.013133,0.033284,0.076594,0.751693,0.0,1.055115e-19,0.030113,0.141600
1,1,0.013623,0.033567,0.088799,0.755367,0.0,7.506947e-03,0.022548,0.125779
2,2,0.014113,0.033962,0.091854,0.756057,0.0,1.670796e-02,0.014695,0.120686
3,3,0.014603,0.034440,0.094908,0.756748,0.0,2.590897e-02,0.006842,0.115593
4,4,0.015093,0.034998,0.098237,0.756307,0.0,3.510823e-02,0.000000,0.110347


Save the portfolios in a TSV file named "efficient_frontier.tsv".
The header row should have columns "index", "return", "risk", and all the asset tickers.
Each data row should have the portfolio index (0-100), the estimated return of the portfolio, the estimated standard deviation (not variance) of the portfolio, and all the asset allocations (which should sum to one).

In [73]:
# YOUR CHANGES HERE

efficient_frontier.to_csv("efficient_frontier.tsv", sep="\t", index=False)

Submit "efficient_frontier.tsv" in Gradescope.

## Part 8: Check Maximum Return Portfolio Stability

Check the stability of the maximum return portfolio by resampling the estimated risk/return model.

Repeat 1000 times -
1. Use `np.random.multivariate_normal` to generate 23 return samples using your previously estimated risks and returns.
2. Estimate the return of each asset using that resampled return history.
3. Check which asset had the highest return in those resampled estimates.

This procedure is a reduced and simplified version of the Michaud resampled efficient frontier procedure that takes uncertainty in the risk model into account.

In [74]:
# YOUR CHANGES HERE

rng = np.random.default_rng(704)

counts = np.zeros(n, dtype=int)
for _ in range(1000):
    sample = rng.multivariate_normal(mu, Sigma, size=23)
    counts[sample.mean(axis=0).argmax()] += 1

max_return_probabilities = pd.DataFrame({
    "asset": TICKERS,
    "probability": counts / 1000,
})
max_return_probabilities

,asset,probability
0,AAPL,0.042
1,SPY,0.009
2,TSLA,0.203
3,MDA.TO,0.690
4,BA,0.017
5,LMT,0.039


Save a file "max_return_probabilities.tsv" with the distribution of highest return assets.
The header row should have columns "asset" and "probability".
There should be a data row for each asset and its sample probability of having the highest return based on those 1000 resampled estimates.


In [75]:
# YOUR CHANGES HERE

max_return_probabilities.to_csv("max_return_probabilities.tsv", sep="\t", index=False)

Submit "max_return_probabilities.tsv" in Gradescope.

## Part 9: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 10: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.